# Chapters 3, 10, 12 — Split-safe temporal windows and training

**DBNet attribution:** Chen et al. (2018), *LiDAR-Video Driving Dataset: Learning Driving Policies Effectively*, CVPR. Official dataset project: https://github.com/driving-behavior/DBNet. Full BibTeX: ../CITATION_DBNet.bib.

**[research]** The research describes an 80/10/10 train/validation/test split. **[implementation]** This notebook demonstrates drive-disjoint split validation, contiguous frame windows, final-frame targets, a weighted two-target RMSE loss, and training of the **temporal model only** using synthetic fused features. It does not claim original DBNet format, original sequence length, or end-to-end fine-tuning.

In [ ]:
%pip install torch
import torch
from torch.utils.data import DataLoader, TensorDataset
from autonomous_driving.sequence_manifest import Observation, build_sequence_manifest, split_windows
from autonomous_driving.temporal_model import build_temporal_model
from autonomous_driving.training import run_epoch

## Construct a synthetic manifest (not DBNet)

Each record represents one synchronized camera/LiDAR pair and two labels. Splits are assigned per drive, not per overlapping temporal window.

In [ ]:
rows = []
for split, drive in [('train','drive_train'), ('val','drive_val'), ('test','drive_test')]:
    for t in range(8):
        rows.append(Observation(f'{drive}-{t}', drive, t, split,
                                f'{drive}/camera_{t}.png', f'{drive}/lidar_{t}.png',
                                float(t), float(10 + t)))
SEQUENCE_LENGTH = 3  # illustrative implementation value, NOT original research value
windows = build_sequence_manifest(rows, sequence_length=SEQUENCE_LENGTH)
parts = split_windows(windows)
print({name: len(part) for name, part in parts.items()})
assert all(len({r.drive_id for r in w}) == 1 for w in windows)

## Synthetic temporal-feature training demonstration

This uses precomputed **random** fused vectors instead of running the image backbones. It therefore trains the temporal head only; it is not an end-to-end DBNet training experiment. The original joint RMSE aggregation is unknown; the weighted average below is a disclosed implementation choice.

In [ ]:
torch.manual_seed(7)
FUSED_FEATURES = 12
train = TensorDataset(torch.randn(12, SEQUENCE_LENGTH, FUSED_FEATURES), torch.randn(12, 2))
val = TensorDataset(torch.randn(4, SEQUENCE_LENGTH, FUSED_FEATURES), torch.randn(4, 2))
train_loader = DataLoader(train, batch_size=4, shuffle=True)
val_loader = DataLoader(val, batch_size=4, shuffle=False)
model = build_temporal_model(FUSED_FEATURES, hidden_size=8)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(2):
    train_loss = run_epoch(model, train_loader, optimizer)
    val_loss = run_epoch(model, val_loader)
    print(f'SYNTHETIC epoch={epoch+1} train_loss={train_loss:.4f} val_loss={val_loss:.4f}')

## Requirements before using DBNet

1. Verify the actual DBNet file schema and image/point-cloud formats.
2. Establish genuine camera/LiDAR timestamp correspondence and label units.
3. Create and archive disjoint drive/session-level train/validation/test assignments. The research's 80/10/10 ratio alone does not prove drive-disjoint splitting was previous.
4. Choose and record sequence length, stride, and target alignment as implementation hyperparameters.
5. Build actual image loading, normalization, LiDAR rasterization, and backbone forward passes; this notebook deliberately does not do these steps.
6. Pin dependencies, record random seeds, and validate on genuine held-out data before reporting metrics.